# LLM을 이용한 그래프 디비 설계

자연어 문서를 입력으로 받아 LLM을 이용해 Neo4j 그래프 데이터베이스의 기본 설계를 생한다.

- 문서에 등장하는 기업, 인물, 서비스, 사건, 기술, 매출 정보 등  내용을 분석하여 그래프 DB에서 사용할 노드(Label), 관계(Relationship), 속성(Property)을 도출하고, 이를 바탕으로 Neo4j에 적재할 수 있는 Cypher 쿼리를 생성한다.
- 쿼리 뿐 아니라 설계 스키마와 설계 근거를 설명하는 명세서도 같이 작성하도록 한다.

- **LLM이 생성한 스키마와 쿼리는 초안으로 사용해야 한다.** 이 초안을 바탕으로 데이터 중복, 관계 방향, Label 명명 규칙, 고유 식별자, 검색 목적 등을 사람이 검토하고 도메인에 맞게 정리한 뒤 사용하는 것이 중요하다.


In [ ]:
DOCUMENTS = [
    "구글은 1995년 스탠퍼드대에서 만난 래리 페이지와 세르게이 브린이 웹페이지의 링크 구조로 중요도를 판단하는 검색엔진 BackRub을 만들면서 시작됐다. 1998년 Sun 공동창업자 앤디 벡톨샤임의 10만 달러 투자 이후 Google Inc.가 공식 출범했고, 검색을 중심으로 광고, Gmail, Maps, Android, YouTube, Cloud, AI로 확장했다. 현재는 알파벳의 핵심 자회사로, 2025년 알파벳 매출은 4,028억 달러였고 Google Services와 Google Cloud가 성장을 이끌었다.",

    "네이버는 이해진이 삼성SDS 사내벤처에서 출발해 1999년 6월 설립한 한국 인터넷 기업이다. 처음에는 검색 포털 네이버와 어린이 포털 쥬니버로 시작했고, 2000년 통합검색, 2001년 검색광고, 지식iN, 블로그, 카페, 웹툰, 쇼핑, 페이, 클라우드, AI로 영역을 넓혔다. 현재는 한국 대표 플랫폼 기업으로 검색·광고·커머스·핀테크·콘텐츠·클라우드가 주력이다. 2025년 연매출은 12조 350억 원, 영업이익은 2조 2,081억 원이었다.",

    "현재의 카카오는 두 다음과 카카오가 합쳐진 회사다. 다음은 1995년 설립되어 한메일, 다음카페, 미디어다음, 검색으로 성장했고, 카카오는 2006년 아이위랩으로 설립된 뒤 2010년 카카오톡을 내놓으며 모바일 메신저 플랫폼으로 커졌다. 2014년 다음과 카카오는 합병을 발표했고, 이후 회사명은 카카오가 되었다. 현재 주력은 카카오톡 기반 광고·커머스, 모빌리티, 페이, 콘텐츠, 음악, 미디어, AI이다. 2025년 매출은 8조 991억 원, 영업이익은 7,320억 원으로 창사 이래 최대였다.",

    "아마존은 제프 베이조스가 1994년 미국 워싱턴주 벨뷰의 차고에서 시작한 회사다. 1995년 온라인 서점 Amazon.com으로 문을 열었고, 이후 책을 넘어 전자상거래 전반, 마켓플레이스, 물류, Prime, Kindle, 영상 스트리밍, 광고로 확장했다. 2006년 시작한 AWS는 클라우드 컴퓨팅 시장의 핵심 축이 되었다. 현재 아마존은 전자상거래, 클라우드, 광고, 구독, 물류를 모두 가진 세계적 기술·유통 기업이다. 2025년 4분기 순매출은 2,134억 달러였고 AWS 매출은 356억 달러였다.",

    "엔비디아는 젠슨 황, 크리스 말라코스키, 커티스 프리엠이 1993년 창업한 반도체 기업이다. 처음에는 PC 그래픽과 게임용 GPU에 집중했고, 1999년 GeForce 256, 2006년 CUDA를 통해 GPU를 범용 병렬계산과 AI 학습에 쓰는 길을 열었다. 이후 데이터센터, AI 가속기, 네트워킹, 자율주행, 로보틱스, Omniverse 등으로 확장했다. 현재는 생성형 AI와 데이터센터 인프라의 핵심 기업이다. 2026년 1분기 회계연도 2027 매출은 816억 달러, 데이터센터 매출은 752억 달러였다.",
    
    "메타는 마크 저커버그가 더스틴 모스코비츠, 크리스 휴스, 에두아르도 세버린과 함께 2004년 2월 4일 하버드에서 Facebook을 시작하며 출발했다. 처음에는 대학생용 소셜 네트워크였지만 빠르게 일반 대중으로 확장했고, 이후 Instagram, WhatsApp, Oculus를 인수했다. 2021년에는 사명을 Meta로 바꾸고 소셜미디어와 광고, 메신저, 메타버스, AI, AR·VR 기기로 사업을 넓혔다. 현재 주력은 Facebook, Instagram, Messenger, WhatsApp을 포함한 Family of Apps 광고 사업이다. 2025년 총매출은 2,009억 달러였다."
]

## 생성 단위

1. **쿼리와 명세서를 한번에 생성한다.**
   - 문서의 내용이 크지 않을 경우 한번에 작성하도록 한다.
2. **단계적으로 나눠서 처리한다.**
   - 문서의 내용이 많을 경우 하나의 거대한 프롬프트보다 단계적으로 작성할 때 품질이 훨씬 높다.
   1. 노드와 관계 추출 및 CREATE
   2. 제약 조건 설계
   3. INDEX 설계
   4. 명세서 작성
   - 각 단계를 Prompt -> Model -> outputparser의 체인으로 구성한 뒤 연결한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

graph_schema_prompt = ChatPromptTemplate.from_template(
    template="""
당신은 Neo4j 그래프 데이터베이스 스키마 설계 전문가이다.

사용자가 제공한 문서를 분석하여 Neo4j에 적합한 그래프 스키마를 설계하고,
그 스키마를 기반으로 Cypher 쿼리를 작성한다.

# 목표

문서의 내용을 바탕으로 다음 작업을 수행한다.

1. 문서에서 주요 개체를 추출하여 노드(Label)를 정의한다.
2. 개체 간 의미 있는 관계를 추출하여 관계 타입(Relationship Type)을 정의한다.
3. 각 노드와 관계에 필요한 속성(Property)을 정의한다.
4. 문서의 내용을 저장하기 위한 Cypher CREATE 쿼리를 작성한다.
5. 각 노드와 관계에 대한 스키마를 표 형태로 출력한다.
6. 데이터 중복 방지와 검색 성능 향상을 위한 제약조건과 인덱스 생성 쿼리를 작성한다.

# 입력 문서

아래 문서를 분석한다.

<문서>
{document}
</문서>

# 작업 지침
- GraphRAG용 스키마를 설계하는 것으로  특히 "LLM이 생성하는 Cypher 쿼리로 탐색하기 쉬운 구조" 여야 한다.

## 1. 노드(Noe) 설계 규칙

- 문서에서 독립적으로 식별 가능한 주요 개체를 노드로 설계한다.
- 다른 개체에서 독립적으로 참조될 수 있어야 한다.
- 그 자체가 직접 검색하거나 탐색하는데 시작점이 될 수있는 개체여야 한다. 
- 사람, 조직, 장소, 상품, 문서, 사건, 개념, 기술, 프로젝트 등은 노드 후보가 될 수 있다.
- 노드 Label은 PascalCase로 작성한다.
  - 예: Person, Company, Product, Project, Document
- 하나의 노드는 최소 하나 이상의 식별 속성을 가져야 한다.
  - 예: name, title, id, code
- 문서에 명시되지 않은 정보는 임의로 만들지 않는다.
- 같은 의미의 개체는 하나의 Label로 통합한다.
  - 예: 기업, 회사, 업체 → Company

## 2. 관계(Relationship) 설계 규칙

- 노드 간 의미 있는 연결을 관계로 설계한다.
- 두 노드 사이에 방향성 있는 어떤 행위, 역할, 연관이어야 한다.
- 관계 타입은 동사 또는 동사구로 작성한다. 
- 모호한 이름은 사용하지 않는다.
- 관계 타입은 대문자와 언더스코어를 사용한다. (대문자 SNAKE_CASE)
  - 예: WORKS_FOR, LOCATED_IN, PRODUCES, PARTICIPATED_IN, BELONGS_TO
- 관계는 방향성을 가지며, 의미상 자연스러운 방향으로 설정한다.
  - 예: (Person)-[:WORKS_FOR]->(Company)
- 관계 자체에 의미 있는 정보가 있으면 관계 속성으로 저장한다.
  - 예: since, role, amount, 
- 단순히 함께 등장했다는 이유만으로 관계를 만들지 않는다.

## 3. 속성(Property) 설계 규칙
- 노드나 관계에 부가적인 정보가 있을 경우 속성으로 설정한다.
- 속성명은 camelCase로 작성한다.
  - 예: name, birthDate, createdAt, totalAmount
- 속성값은 문서에 있는 내용만 사용한다.
- 날짜, 금액, 수량, 상태, 역할 등은 가능한 속성으로 분리한다.
- 긴 설명문은 description, summary, content 등의 속성으로 저장할 수 있다.
- 배열이 필요한 경우 리스트 속성을 사용할 수 있다.
  - 예: tags: ["AI", "GraphDB"]
- 속성 타입 규칙
  - 문자열: String
  - 정수: Integer  
  - 실수: Float
  - 불리언: Boolean
  - 날짜: Date (ISO 형식 문자열)
  - 목록: List

## 4. Cypher CREATE 쿼리 작성 규칙

- 문서에서 추출한 실제 데이터를 기반으로 `CREATE` 쿼리를 작성한다.
- 노드와 관계를 함께 생성하는 Cypher 쿼리를 작성한다.
- 변수명은 소문자로 작성한다.
  - 예: person1, company1, project1
- 문자열 값은 작은따옴표 또는 큰따옴표 중 하나로 일관되게 사용한다.
- 문서에 없는 값은 생성하지 않는다.
- 동일한 노드가 여러 관계에서 사용될 경우 하나의 변수로 재사용한다.
- 여러 개의 독립적인 데이터가 있으면 가독성을 위해 여러 개의 CREATE 문으로 나누어 작성할 수 있다.

## 5. 제약조건 작성 규칙

- 각 주요 노드에는 가능한 경우 고유 식별 속성에 대해 UNIQUE 제약조건을 제안한다.
- 제약조건 이름은 명확하게 작성한다.
  - 예: person_name_unique, company_name_unique
- Neo4j 5.x 이상 문법을 사용한다.

예시:

CREATE CONSTRAINT person_name_unique IF NOT EXISTS
FOR (p:Person)
REQUIRE p.name IS UNIQUE;

## 6. 인덱스 작성 규칙

- 자주 검색될 가능성이 높은 속성에 대해 인덱스를 제안한다.
- 이름, 제목, 코드, 날짜, 상태, 카테고리 등은 인덱스 후보가 될 수 있다.
- UNIQUE 제약조건이 걸린 속성에는 별도 일반 인덱스를 중복으로 만들지 않는다.
- Neo4j 5.x 이상 문법을 사용한다.

예시:

CREATE INDEX movie_title_index IF NOT EXISTS
FOR (m:Movie)
ON (m.title);

## 7. 출력 형식

출력은 markdown으로 작성하며 반드시 아래 내용으로 구성한다.

<출력 구조>
# 1. 추출된 그래프 스키마 요약

문서에서 어떤 기준으로 노드와 관계를 설계 했는지 간단히 설명한다.

---

# 2. 노드 스키마 예

| Label | 설명 | 주요 속성 | 고유 식별 속성 |
|---|---|---|---|
| Person | 사람 | name, role, birthDate | name |

---

# 3. 관계 스키마 예

| 관계 타입 | 시작 노드 | 종료 노드 | 설명 | 관계 속성 |
|---|---|---|---|---|
| WORKS_FOR | Person | Company | 사람이 회사에 소속됨 | role, since |

---

# 4. Cypher CREATE 쿼리

```cypher
// 문서 내용을 기반으로 노드와 관계를 생성하는 쿼리
```

# 5. 제약조건 생성 쿼리
```cypher
// UNIQUE 제약조건 생성 쿼리
```

# 6. Index 생성 쿼리
```cypher
// 검색 성능 향상을 위한 인덱스 생성 쿼리
```

7. 설계 근거

* 왜 해당 개체를 노드로 설계했는지 설명한다.
* 왜 해당 연결을 관계로 설계했는지 설명한다.
* 어떤 속성을 식별자 또는 검색 대상으로 판단했는지 설명한다.
* 문서에 정보가 부족하여 확정하기 어려운 부분이 있다면 명시한다.
</출력 구조>
---

## 8. 주의사항

- 문서에 없는 내용을 추측해서 만들지 않는다.
- Neo4j에서 실행 가능한 Cypher 문법으로 작성한다.
- Label, Relationship Type, Property 이름은 일관된 네이밍 규칙을 따른다.
- 가능한 한 단순하고 명확한 그래프 구조를 우선한다.
- 지나치게 많은 노드와 관계를 만들지 말고, 실제 질의에 유용한 구조를 설계한다.
""")

In [ ]:
##########################################
# 지식 그래프로 변환
##########################################
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()


model = ChatOpenAI(model="gpt-5.5")

chain = graph_schema_prompt | model

def convert_to_graph_doc(docs:list[str], labels: list[str]=None, relationship:list[str]=None) -> list[str]:
    """문서들을 받아서 db schema(index, constraint) 와 data 생성(Node, Relationship) 생성 cypher query 만드는 함수

    Args:
        docs (list[str]): 쿼리를 만들 문서들
        labels (list[str], optional): 추출할 Node의 Label들. Defaults to None.
        relationship (list[str], optional): 추출할 관계들. Defaults to None.

    Returns:
        list[str]: 문서별로 생성된 결과를 리스트에 담아서 반환한다.
    """
    graph_docs = []
    for doc in docs:
        result = chain.invoke(
                {
                    "labels": labels if labels else "None",
                    "relationship": relationship if relationship else "None",
                    "document": doc
                }
        )
        graph_docs.append(result)
    
    return graph_docs

In [ ]:
graph_docs = convert_to_graph_doc(DOCUMENTS)

In [ ]:
from IPython.display import Markdown

Markdown(graph_docs[0].content)


# 1. 추출된 그래프 스키마 요약

문서에서 독립적으로 식별 가능한 사람, 회사, 조직, 서비스/제품, 사업 부문, 기술 개념, 재무 실적을 노드로 설계했다.

- 사람: 래리 페이지, 세르게이 브린, 앤디 벡톨샤임
- 회사: Google, Alphabet, Sun
- 조직/장소: Stanford University
- 서비스/제품/사업 영역: BackRub, Search, Advertising, Gmail, Maps, Android, YouTube, Cloud, AI
- 사업 부문: Google Services, Google Cloud
- 기술 개념: 웹페이지 링크 구조 기반 중요도 판단
- 재무 실적: 2025년 Alphabet 매출

관계는 문서에 명시된 의미를 기준으로 설계했다. 예를 들어, 래리 페이지와 세르게이 브린이 BackRub을 만들었다는 내용은 `CREATED` 관계로, Google이 Alphabet의 핵심 자회사라는 내용은 `SUBSIDIARY_OF` 관계로 표현했다.

---

# 2. 노드 스키마 예

| Label | 설명 | 주요 속성 | 고유 식별 속성 |
|---|---|---|---|
| Person | 문서에 등장하는 인물 | name, koreanName | name |
| Company | 기업 또는 회사 | name, koreanName, legalName, launchYear, description | name |
| Organization | 대학 등 조직 또는 장소 성격의 기관 | name, koreanName, type | name |
| Offering | 제품, 서비스, 사업 영역 또는 기술 분야 | name, type, description | name |
| BusinessSegment | 기업의 사업 부문 | name, description | name |
| TechnologyConcept | 기술적 개념 또는 방법론 | name, description | name |
| Event | 만남 등 특정 사건 | name, type, year | name |
| FinancialResult | 특정 조직의 특정 연도 재무 지표 | organizationName, metricType, year, amountUsdHundredMillion, currency, originalAmount | organizationName, metricType, year |

---

# 3. 관계 스키마 예

| 관계 타입 | 시작 노드 | 종료 노드 | 설명 | 관계 속성 |
|---|---|---|---|---|
| PARTICIPATED_IN | Person | Event | 사람이 사건에 참여함 | role |
| OCCURRED_AT | Event | Organization | 사건이 특정 장소 또는 기관에서 발생함 | 없음 |
| CREATED | Person | Offering | 사람이 제품 또는 서비스를 만듦 | year |
| STARTED_WITH | Company | Offering | 회사의 시작이 특정 제품 또는 서비스에서 비롯됨 | 없음 |
| USES_METHOD | Offering | TechnologyConcept | 제품이 특정 기술적 방법을 사용함 | 없음 |
| CO_FOUNDED | Person | Company | 사람이 회사를 공동창업함 | 없음 |
| INVESTED_IN | Person | Company | 사람이 회사에 투자함 | year, amountUsd, currency |
| FOCUSED_ON | Company | Offering | 회사가 특정 영역을 중심으로 함 | 없음 |
| EXPANDED_TO | Company | Offering | 회사가 특정 제품, 서비스, 분야로 확장함 | 없음 |
| SUBSIDIARY_OF | Company | Company | 회사가 다른 회사의 자회사임 | status, role |
| HAS_BUSINESS_SEGMENT | Company | BusinessSegment | 회사가 특정 사업 부문을 가짐 | 없음 |
| REPORTED_FINANCIAL_RESULT | Company | FinancialResult | 회사가 특정 재무 실적을 기록함 | 없음 |
| DROVE_GROWTH | BusinessSegment | Company | 사업 부문이 회사 성장을 이끎 | year |

---

# 4. Cypher CREATE 쿼리

```cypher
CREATE
  (larry:Person {
    name: 'Larry Page',
    koreanName: '래리 페이지'
  }),
  (sergey:Person {
    name: 'Sergey Brin',
    koreanName: '세르게이 브린'
  }),
  (andy:Person {
    name: 'Andy Bechtolsheim',
    koreanName: '앤디 벡톨샤임'
  }),

  (google:Company {
    name: 'Google',
    koreanName: '구글',
    legalName: 'Google Inc.',
    launchYear: 1998,
    description: '알파벳의 핵심 자회사'
  }),
  (alphabet:Company {
    name: 'Alphabet',
    koreanName: '알파벳'
  }),
  (sun:Company {
    name: 'Sun'
  }),

  (stanford:Organization {
    name: 'Stanford University',
    koreanName: '스탠퍼드대',
    type: 'University'
  }),

  (meeting_1995:Event {
    name: '래리 페이지와 세르게이 브린의 스탠퍼드대 만남',
    type: 'Meeting',
    year: 1995
  }),

  (backrub:Offering {
    name: 'BackRub',
    type: 'SearchEngine',
    description: '웹페이지의 링크 구조로 중요도를 판단하는 검색엔진'
  }),
  (search:Offering {
    name: 'Search',
    koreanName: '검색',
    type: 'Service'
  }),
  (advertising:Offering {
    name: 'Advertising',
    koreanName: '광고',
    type: 'BusinessArea'
  }),
  (gmail:Offering {
    name: 'Gmail',
    type: 'Service'
  }),
  (maps:Offering {
    name: 'Maps',
    type: 'Service'
  }),
  (android:Offering {
    name: 'Android',
    type: 'Platform'
  }),
  (youtube:Offering {
    name: 'YouTube',
    type: 'Service'
  }),
  (cloud:Offering {
    name: 'Cloud',
    type: 'TechnologyArea'
  }),
  (ai:Offering {
    name: 'AI',
    type: 'TechnologyArea'
  }),

  (link_structure:TechnologyConcept {
    name: '웹페이지 링크 구조 기반 중요도 판단',
    description: '웹페이지의 링크 구조로 중요도를 판단하는 방법'
  }),

  (google_services:BusinessSegment {
    name: 'Google Services'
  }),
  (google_cloud_segment:BusinessSegment {
    name: 'Google Cloud'
  }),

  (alphabet_revenue_2025:FinancialResult {
    organizationName: 'Alphabet',
    metricType: 'revenue',
    year: 2025,
    amountUsdHundredMillion: 4028,
    currency: 'USD',
    originalAmount: '4,028억 달러'
  }),

  (larry)-[:PARTICIPATED_IN {role: '만남 참여자'}]->(meeting_1995),
  (sergey)-[:PARTICIPATED_IN {role: '만남 참여자'}]->(meeting_1995),
  (meeting_1995)-[:OCCURRED_AT]->(stanford),

  (larry)-[:CREATED {year: 1995}]->(backrub),
  (sergey)-[:CREATED {year: 1995}]->(backrub),
  (google)-[:STARTED_WITH]->(backrub),
  (backrub)-[:USES_METHOD]->(link_structure),

  (andy)-[:CO_FOUNDED]->(sun),
  (andy)-[:INVESTED_IN {
    year: 1998,
    amountUsd: 100000,
    currency: 'USD'
  }]->(google),

  (google)-[:FOCUSED_ON]->(search),
  (google)-[:EXPANDED_TO]->(advertising),
  (google)-[:EXPANDED_TO]->(gmail),
  (google)-[:EXPANDED_TO]->(maps),
  (google)-[:EXPANDED_TO]->(android),
  (google)-[:EXPANDED_TO]->(youtube),
  (google)-[:EXPANDED_TO]->(cloud),
  (google)-[:EXPANDED_TO]->(ai),

  (google)-[:SUBSIDIARY_OF {
    status: 'current',
    role: 'core subsidiary'
  }]->(alphabet),

  (alphabet)-[:HAS_BUSINESS_SEGMENT]->(google_services),
  (alphabet)-[:HAS_BUSINESS_SEGMENT]->(google_cloud_segment),
  (alphabet)-[:REPORTED_FINANCIAL_RESULT]->(alphabet_revenue_2025),

  (google_services)-[:DROVE_GROWTH {year: 2025}]->(alphabet),
  (google_cloud_segment)-[:DROVE_GROWTH {year: 2025}]->(alphabet);
```

---

# 5. 제약조건 생성 쿼리

```cypher
CREATE CONSTRAINT person_name_unique IF NOT EXISTS
FOR (p:Person)
REQUIRE p.name IS UNIQUE;

CREATE CONSTRAINT company_name_unique IF NOT EXISTS
FOR (c:Company)
REQUIRE c.name IS UNIQUE;

CREATE CONSTRAINT organization_name_unique IF NOT EXISTS
FOR (o:Organization)
REQUIRE o.name IS UNIQUE;

CREATE CONSTRAINT offering_name_unique IF NOT EXISTS
FOR (o:Offering)
REQUIRE o.name IS UNIQUE;

CREATE CONSTRAINT business_segment_name_unique IF NOT EXISTS
FOR (b:BusinessSegment)
REQUIRE b.name IS UNIQUE;

CREATE CONSTRAINT technology_concept_name_unique IF NOT EXISTS
FOR (t:TechnologyConcept)
REQUIRE t.name IS UNIQUE;

CREATE CONSTRAINT event_name_unique IF NOT EXISTS
FOR (e:Event)
REQUIRE e.name IS UNIQUE;

CREATE CONSTRAINT financial_result_identity_unique IF NOT EXISTS
FOR (f:FinancialResult)
REQUIRE (f.organizationName, f.metricType, f.year) IS UNIQUE;
```

---

# 6. Index 생성 쿼리

```cypher
CREATE INDEX person_korean_name_index IF NOT EXISTS
FOR (p:Person)
ON (p.koreanName);

CREATE INDEX company_korean_name_index IF NOT EXISTS
FOR (c:Company)
ON (c.koreanName);

CREATE INDEX company_launch_year_index IF NOT EXISTS
FOR (c:Company)
ON (c.launchYear);

CREATE INDEX organization_type_index IF NOT EXISTS
FOR (o:Organization)
ON (o.type);

CREATE INDEX offering_type_index IF NOT EXISTS
FOR (o:Offering)
ON (o.type);

CREATE INDEX event_year_index IF NOT EXISTS
FOR (e:Event)
ON (e.year);

CREATE INDEX financial_result_year_index IF NOT EXISTS
FOR (f:FinancialResult)
ON (f.year);

CREATE INDEX financial_result_metric_type_index IF NOT EXISTS
FOR (f:FinancialResult)
ON (f.metricType);
```

---

# 7. 설계 근거

## 노드 설계 근거

- `Person`은 래리 페이지, 세르게이 브린, 앤디 벡톨샤임처럼 독립적으로 식별 가능한 인물을 표현하기 위해 설계했다.
- `Company`는 Google, Alphabet, Sun처럼 기업 단위로 식별되는 개체를 표현하기 위해 설계했다.
- `Organization`은 스탠퍼드대처럼 회사는 아니지만 사건이 발생한 주요 기관이므로 별도 노드로 설계했다.
- `Offering`은 BackRub, Gmail, Maps, Android, YouTube, Cloud, AI 등 Google의 제품, 서비스, 기술 영역 확장을 표현하기 위해 설계했다.
- `BusinessSegment`는 문서에서 Google Services와 Google Cloud가 Alphabet의 성장을 이끌었다고 명시되어 있어, 일반 서비스가 아니라 사업 부문으로 분리했다.
- `TechnologyConcept`는 BackRub의 핵심 원리인 “웹페이지 링크 구조로 중요도를 판단”하는 방식을 표현하기 위해 설계했다.
- `FinancialResult`는 2025년 Alphabet 매출 4,028억 달러라는 정량적 재무 정보를 구조화하기 위해 설계했다.
- `Event`는 1995년 스탠퍼드대에서의 만남처럼 시점과 장소가 중요한 사건을 표현하기 위해 사용했다.

## 관계 설계 근거

- 래리 페이지와 세르게이 브린이 BackRub을 만들었다는 내용은 `CREATED` 관계로 표현했다.
- Google이 BackRub에서 시작됐다는 내용은 `STARTED_WITH` 관계로 표현했다.
- BackRub이 웹페이지 링크 구조를 활용했다는 내용은 `USES_METHOD` 관계로 표현했다.
- 앤디 벡톨샤임이 Sun 공동창업자라는 내용은 `CO_FOUNDED` 관계로 표현했다.
- 앤디 벡톨샤임의 10만 달러 투자는 `INVESTED_IN` 관계와 관계 속성 `amountUsd`, `year`, `currency`로 표현했다.
- Google이 검색을 중심으로 했다는 내용은 `FOCUSED_ON` 관계로 표현했다.
- Google이 광고, Gmail, Maps, Android, YouTube, Cloud, AI로 확장했다는 내용은 `EXPANDED_TO` 관계로 표현했다.
- Google이 Alphabet의 핵심 자회사라는 내용은 `SUBSIDIARY_OF` 관계로 표현했다.
- Alphabet의 2025년 매출 정보는 `REPORTED_FINANCIAL_RESULT` 관계로 연결했다.
- Google Services와 Google Cloud가 성장을 이끌었다는 내용은 `DROVE_GROWTH` 관계로 표현했다.

## 식별자 및 검색 대상 판단

- 사람, 회사, 조직, 서비스, 사업 부문은 이름으로 식별 가능하므로 `name`에 UNIQUE 제약조건을 제안했다.
- `FinancialResult`는 단일 이름보다 `organizationName`, `metricType`, `year` 조합이 자연스러운 식별자이므로 복합 UNIQUE 제약조건을 제안했다.
- 연도 기반 검색 가능성이 높은 `Event.year`, `FinancialResult.year`, `Company.launchYear`에는 인덱스를 제안했다.
- 서비스 유형별 검색을 위해 `Offering.type`에도 인덱스를 제안했다.

## 정보 부족 또는 확정이 어려운 부분

- 문서에는 정확한 월일 정보가 없으므로 날짜 타입 `Date` 대신 `year` 정수 속성을 사용했다.
- Google의 제품별 출시 연도는 문서에 없으므로 생성하지 않았다.
- 스탠퍼드대에서 만났다는 사실은 명시되어 있지만, 두 인물이 당시 학생이었는지 등의 소속 관계는 문서에 없으므로 만들지 않았다.
- Sun의 정식 회사명은 문서에 “Sun”으로만 등장하므로 `Sun Microsystems`로 확장하지 않고 `Sun`으로 저장했다.
- 4,028억 달러는 원문 표현을 보존하기 위해 `originalAmount`에 저장하고, 수치 검색을 위해 `amountUsdHundredMillion: 4028`로 함께 저장했다.